In [1]:
#from langchain_openai import ChatOpenAI
#from langchain_core.prompts import ChatPromptTemplate
#from langchain_core.runnables import RunnableSequence, RunnablePassthrough, RunnableParallel
#from langchain_core.output_parsers import StrOutputParser
import os

In [2]:
with open("api_key.txt") as archivo:
  apikey = archivo.read()
os.environ["OPENAI_API_KEY"] = apikey

## Métodos principales de la interfaz Runnable

``` .invoke(input)```
- Procesa una única entrada y devuelve la salida correspondiente.

In [3]:
from langchain_openai import ChatOpenAI
from langchain.chat_models import init_chat_model

# llm = ChatOpenAI()
llm = init_chat_model("gpt-4o-mini", model_provider="openai")

response = llm.invoke("¿Cuál es la capital de Francia?")
print(response.content)

La capital de Francia es París.


``` .batch(inputs: List)```
- Procesa múltiples entradas en paralelo, devolviendo una lista de resultados en el mismo orden que las entradas.

In [4]:
inputs = ["¿Capital de España?", "¿Capital de Alemania?"]
responses = llm.batch(inputs)
print(responses[0].content, responses[1].content)

La capital de España es Madrid. La capital de Alemania es Berlín.


``` .stream(input)```
- Devuelve un generador que produce la salida de manera progresiva, útil para modelos que generan texto en tiempo real.

Se recomienda su uso cuando el texto es muy largo, porque te manda como paquete de datos.

In [6]:
i = 1
for chunk in llm.stream("Dime el poema sobre la luna"):
    print(chunk.content, end="")
    #print(str(i)+")"+chunk.content+" ", end="")
    #i += 1

Claro, aquí tienes un poema inspirado en la luna:

**En la noche plateada**

Bajo el manto estrellado,  
la luna asoma su rostro,  
un faro en el oscuro cielo,  
susurro de sueños en el anochecer.

Su luz, un reflejo de misterio,  
acaricia la superficie del río,  
dibujando senderos de plata,  
guiando a los navegantes perdidos.

En sus cráteres guardan secretos  
de amores antiguos y desvelos,  
sutil testigo de promesas,  
brillando en corazones anhelantes.

Las sombras se alargan suavemente,  
los árboles danzan en su brillo,  
los susurros de la brisa murmuran  
canciones de un tiempo olvidado.

Oh luna, musa de poetas,  
en tus fases cuento mis días,  
te busco en cada ocaso,  
en cada amanecer que se olvida.

Eres la eterna compañera  
de los sueños y las esperanzas,  
un destello en la memoria  
de quienes miran al cielo y esperan.

Espero que te haya gustado. Si quieres otro tema o estilo, ¡dímelo!

## Composicion de runnables

### **1**. RunnableSequence (Secuencia)

El StrOutputParser() sirve para convertir la respuesta del modelo en texto limpio.

Sin parser, llm devuelve un objeto tipo AIMessage.


Con parser, obtienes solo un str.

In [7]:
from langchain_classic.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_classic.schema.runnable import RunnableSequence

# 1. Crear los componentes
prompt = ChatPromptTemplate.from_template("Indica en una palabra de que genero es la pelicula: {pelicula}")
llm = init_chat_model("gpt-4o-mini", model_provider="openai")
parser = StrOutputParser()

# 2. Componer en secuencia
pipeline = RunnableSequence(prompt | llm | parser)

# 3. Ejecutar
output = pipeline.invoke({"pelicula": "La historia sin fin"})
print(output)


Fantasía.


### **2**. RunnableParallel (Paralelo)

In [8]:
from langchain_classic.schema.runnable import RunnableParallel

parallel = RunnableParallel({
    "mayus": lambda x: x.upper(),
    "longitud": lambda x: len(x),
})

print(parallel.invoke("hola mundo"))

{'mayus': 'HOLA MUNDO', 'longitud': 10}


### **3**. Composición anidada y estructuras complejas

In [9]:
from langchain_classic.schema.runnable import RunnablePassthrough
prompt = ChatPromptTemplate.from_template("Indica en una palabra de que genero es la pelicula: {pelicula}")
llm = ChatOpenAI()
parser = StrOutputParser()

workflow = RunnableSequence(
    RunnableParallel({
        "original": RunnablePassthrough(),
        "genero": prompt | llm | parser,
    }),
    lambda x: f"Original: {x['original']} | Genero: {x['genero']}"
)

print(workflow.invoke({"pelicula": "Terminator"}))


Original: {'pelicula': 'Terminator'} | Genero: Ciencia ficción.


## Otros runnables

### **1**. RunnableLambda

In [10]:
from langchain_classic.schema.runnable import RunnableLambda

mayusculas = RunnableLambda(lambda x: x.upper())
print(mayusculas.invoke("hola mundo"))  # HOLA MUNDO

HOLA MUNDO


In [11]:
extract_nombre = RunnableLambda(lambda x: x["apellido"])
print(extract_nombre.invoke({"nombre": "Miguel", "apellido":"Cotrina"}))  # Miguel

Cotrina


### **2**. RunnablePassthrough

In [12]:
from langchain_classic.schema.runnable import RunnablePassthrough

passthrough = RunnablePassthrough()
print(passthrough.invoke("texto directo"))

texto directo


### **3**. RunnableMap

In [13]:
from langchain_classic.schema.runnable import RunnableMap, RunnableLambda

procesador = RunnableMap({
    "texto": RunnableLambda(lambda x: x["texto"].upper()),
    "longitud": RunnableLambda(lambda x: len(x["texto"]))
})

input_data = {"texto": "langchain es potente"}
print(procesador.invoke(input_data))

{'texto': 'LANGCHAIN ES POTENTE', 'longitud': 20}


## Recordatorio de lambda en python

- Sintaxis: lambda parametros: expresión

In [14]:
def mi_funcion(x):
    return x + 1

# Equivalente en lambda:
r = lambda x: x + 1
print(r.__call__(5))

6


In [15]:
a = lambda x: x * 2            # Multiplica por 2
b = lambda x: x["nombre"]      # Devuelve el valor de la clave 'nombre'
c = lambda x: x.lower().strip()  # Limpia texto
d = lambda x: len(x.split())   # Cuenta palabras

print(a.__call__(5), d.__call__("Hola mundo"))

10 2
